# Assignment – Week 4
## Logistic Regression: R → Python Bridge

**Course:** Data Science  
**Submission:** Jupyter Notebook (.ipynb)  
**Dataset:** `housing.csv`

---

## Learning Objectives

By the end of this assignment, you should be able to:

- Fit and interpret **logistic regression models**
- Translate workflows from **R (glm / tidymodels)** to **Python**
- Interpret **odds ratios**
- Evaluate classification models using **Accuracy and ROC–AUC**
- Reflect on the difference between **statistical inference vs prediction**

---

## Dataset Description

The dataset contains **600 housing listings**.

| Variable | Description |
|---|---|
| listing_id | Unique identifier |
| price | Sale price of the house |
| size | House size (square footage) |
| bedrooms | Number of bedrooms |
| neighborhood | Location category |
| type | Housing type (SingleFamily, Townhouse, MultiFamily) |

---
## Setup — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, RocCurveDisplay

df = pd.read_csv('housing.csv')
print(df.shape)
df.head()

---
## Step 0 – Create a Binary Outcome

For classification, convert `price` into a binary variable:

- `1` → expensive homes (above median price)
- `0` → less expensive homes (at or below median price)

In [ ]:
df['high_price'] = (df['price'] > df['price'].median()).astype(int)

print('Median price:', df['price'].median())
print(df['high_price'].value_counts())

---
## Part E – Explore the Dataset

Before modeling, create at least three exploratory plots and discuss one interesting pattern.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Exploratory Data Analysis — Housing Dataset', fontsize=14)

# 1. Histogram of price
axes[0, 0].hist(df['price'].dropna(), bins=30, color='#534AB7', alpha=0.7, edgecolor='white')
axes[0, 0].set_title('Distribution of sale price')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# 2. Scatter: size vs price
axes[0, 1].scatter(df['size'], df['price'], alpha=0.3, s=18, color='#0F6E56')
axes[0, 1].set_title('Size vs. price')
axes[0, 1].set_xlabel('Square footage')
axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# 3. Boxplot: price by neighborhood
neighborhoods = df['neighborhood'].dropna().unique()
data_by_hood = [df.loc[df['neighborhood'] == n, 'price'].dropna() for n in neighborhoods]
axes[1, 0].boxplot(data_by_hood, labels=neighborhoods, patch_artist=True,
                   boxprops=dict(facecolor='#FAC775', color='#BA7517'),
                   medianprops=dict(color='#2C2C2A', linewidth=2))
axes[1, 0].set_title('Price by neighborhood')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[1, 0].tick_params(axis='x', rotation=15)

# 4. Bar chart: housing type
type_counts = df['type'].value_counts()
axes[1, 1].bar(type_counts.index, type_counts.values, color='#D85A30', alpha=0.8, edgecolor='white')
axes[1, 1].set_title('Housing type counts')
axes[1, 1].set_xlabel('Type')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()

### Your Response — Interesting Pattern

*Discuss one interesting pattern you observed in the plots above.*

> **Your answer here:**

---
## Part A – Logistic Regression for Inference

Fit a logistic regression model using `statsmodels` to understand which variables are associated with a house being expensive.

In [ ]:
df_clean = df.dropna(subset=['high_price', 'size', 'bedrooms', 'neighborhood'])

logit_model = smf.logit(
    'high_price ~ size + bedrooms + C(neighborhood)',
    data=df_clean
).fit()

print(logit_model.summary())

In [ ]:
# Build odds ratio table
odds_table = pd.DataFrame({
    'Coefficient': logit_model.params,
    'Odds Ratio':  np.exp(logit_model.params),
    'p-value':     logit_model.pvalues
}).round(4)

odds_table['Significant'] = odds_table['p-value'] < 0.05
print(odds_table)

### Your Response — Short Analysis

Answer both questions using the table above.

**Which predictors appear statistically significant?**

> **Your answer here:**

**Which neighborhood has higher odds of expensive homes?**

> **Your answer here:**

---
## Part B – Interpretation in Plain Language

Choose **one variable** from the odds ratio table above and explain what its odds ratio means to someone without a statistics background.

Example template:
> *"For every additional bedroom, the odds of a house being expensive increase by ___."*

### Your Response

> **Your answer here:**

---
## Part C – Prediction Workflow (tidymodels → scikit-learn)

### 1. Train / Test Split

In [ ]:
features = ['size', 'bedrooms', 'neighborhood', 'type']
target   = 'high_price'

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training rows: {len(X_train)}  |  Test rows: {len(X_test)}')

### 2. Fit Logistic Regression

In [ ]:
numeric_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median'))
])

categorical_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipe,     ['size', 'bedrooms']),
    ('cat', categorical_pipe, ['neighborhood', 'type'])
])

clf = Pipeline(steps=[
    ('prep', preprocessor),
    ('clf',  LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)
print('Model fitted successfully.')

### 3. Evaluate the Model

In [ ]:
y_pred      = clf.predict(X_test)
y_pred_prob = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_prob)

print(f'Accuracy : {acc:.4f}')
print(f'ROC-AUC  : {auc:.4f}')

In [ ]:
# Optional: ROC curve
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=ax, color='#534AB7')
ax.plot([0, 1], [0, 1], linestyle='--', color='#888780', linewidth=1)
ax.set_title('ROC curve — logistic regression')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Part D – Model Understanding

### Accuracy vs ROC–AUC

Why might **ROC–AUC** sometimes be preferred over **accuracy**? Answer in 2–3 sentences.

### Your Response

> **Your answer here:**

### Inference vs Prediction

Which modeling approach would you choose for each task, and why?

| Task | Approach | Why |
|---|---|---|
| Policy analysis | ? | ? |
| Prediction tasks | ? | ? |

### Your Response

> **Your answer here:**